<a href="https://colab.research.google.com/github/koko2585/fault-diagnosis/blob/main/02_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv(r'/content/sample_data/feature_time_48k_2048_load_1.csv')

In [ ]:
X = df.drop(columns='fault')
y = df['fault']

X.shape, y.shape

((2300, 9), (2300,))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

Logistic Regression

In [ ]:
model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train_scaled, y_train)

y_pred = model_lr.predict(X_test_scaled)

print('Logistic Regression')
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print()

Logistic Regression
Accuracy: 0.9282608695652174
              precision    recall  f1-score   support

           0       0.91      0.93      0.92        46
           1       0.85      0.90      0.87        50
           2       0.85      0.82      0.83        49
           3       0.97      1.00      0.99        37
           4       1.00      1.00      1.00        47
           5       0.95      1.00      0.97        39
           6       0.93      0.95      0.94        39
           7       0.98      1.00      0.99        54
           8       0.86      0.78      0.82        54
           9       1.00      0.96      0.98        45

    accuracy                           0.93       460
   macro avg       0.93      0.93      0.93       460
weighted avg       0.93      0.93      0.93       460




In [ ]:
tree_models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42)
}

for name, model in tree_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print()

Random Forest
Accuracy: 0.967391304347826
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        46
           1       0.92      0.94      0.93        50
           2       0.96      0.90      0.93        49
           3       1.00      1.00      1.00        37
           4       1.00      1.00      1.00        47
           5       0.95      1.00      0.97        39
           6       1.00      1.00      1.00        39
           7       1.00      1.00      1.00        54
           8       0.93      0.93      0.93        54
           9       1.00      1.00      1.00        45

    accuracy                           0.97       460
   macro avg       0.97      0.97      0.97       460
weighted avg       0.97      0.97      0.97       460


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000205 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2295
[L

In [ ]:
model_dnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(len(encoder.classes_), activation='softmax')
])

model_dnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_dnn.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_pred = model_dnn.predict(X_test_scaled).argmax(axis=1)

print("DNN")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
DNN
Accuracy: 0.9043478260869565
              precision    recall  f1-score   support

           0       0.90      0.93      0.91        46
           1       0.84      0.86      0.85        50
           2       0.77      0.82      0.79        49
           3       0.95      1.00      0.97        37
           4       1.00      0.98      0.99        47
           5       0.95      1.00      0.97        39
           6       0.88      0.92      0.90        39
           7       0.98      1.00      0.99        54
           8       0.80      0.65      0.71        54
           9       1.00      0.96      0.98        45

    accuracy                           0.90       460
   macro avg       0.91      0.91      0.91       460
weighted avg       0.90      0.90      0.90       460



|model|  | precision | recall | f1-score |
|:---:| :---  | :--- | ---: | ---: |
|logistic regression|   acc |   |  |  0.93 |
||    macro avg   | 0.93   | 0.93  | 0.93 |
||   weighted avg    | 0.93   | 0.93   | 0.93  |
|random forest|   acc |   |  |  0.97  |
||    macro avg   | 0.97   | 0.97  | 0.97 |
||   weighted avg    | 0.97   | 0.97   | 0.97  |
|lightgbm|   acc |   |  |  0.96  |
||    macro avg   | 0.96   | 0.96  | 0.96 |
||   weighted avg    | 0.96   | 0.96   | 0.96  |
|xgboost|   acc    |   |  |  0.95 |
||    macro avg   | 0.95  | 0.96  | 0.95 |
||     weighted avg  | 0.95  | 0.95  | 0.95 |
|DNN|   acc    |   |  |  0.90 |
||    macro avg   | 0.91  | 0.91  | 0.91 |
||     weighted avg  | 0.90  | 0.90  | 0.90 |

Based on the baseline results, Random Forest and LightGBM achieved
the highest test-set performance among the evaluated models.

Random Forest achieved an F1-score of 0.97, while LightGBM achieved
0.96. Logistic Regression and XGBoost achieved 0.93 and 0.95,
respectively.

These models will be considered for further evaluation and tuning.